<a href="https://colab.research.google.com/github/mikahamelin-gif/IS-4487-final-project-nvidia/blob/main/IS_4487_Final_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Read Me:
- Before working, click the arrow by "run all" and select "restart session and run all" to wipe the history and run all cells in order
- Commit changes to GitHub by saving after ***every time*** you edit it

# IS 4487 Final Project - NVIDIA

### Mikayla Hamelin, Beck Feely, Nathaniel Elbaum, Billy Ortiz


### Summary of business problem:

NVIDIA has to order chips from its manufacturing partners months before it sells them, while demand from AI data centers is growing fast and hard to predict. If it underestimates demand, it misses sales and customers turn to competitors. If it overestimates, it’s left with costly inventory it can’t sell. Our project forecasts how many AI chips NVIDIA will ship next quarter, measured in units or H100 equivalents, so it can plan production and capacity ahead of time.


## Business Problem & Context

### Description of the organization and the business challenge:

AI is the world's fastest growing market and technology, and this boom is supported largely by one company's technology: NVIDIA. NVIDIA designs graphics processing units (GPUs) and the software that runs them, and its chips have become the main hardware used to train and run AI models becuase of their ability to perform parallel processing, or runinng thousands of calculations at once. NVIDIA orders the chips they design from manufacturing partners months in advance. The challenge is predicting how many chips it will need when AI demand is growing quickly and changing from quarter to quarter.

### Why this problem matters (business impact)

Because of the nature of this industry, NVIDIA is in a precarious position. If NVIDIA orders too few chips, it misses sales and risks pushing its key buyers (cloud providers, AI companies, and other data center operators) toward competitors like AMD or toward building their own chips, as Google and Amazon already do. NVIDIA leads the AI chip market, so many other companies' operations and valuations depend on its supply, and some would argue the AI industry as a whole does too. If it orders too many, it ends up with expensive inventory that loses value as newer chips come out. Since production space has to be reserved well ahead of time, a better forecast helps NVIDIA plan capacity, manage costs, and keep its biggest customers and the broader AI industry supplied.

###Brief description of relevant industry or market factors
*  **Big tech spending**: Most demand comes from a few large companies like Microsoft,Google, Amazon and Meta, so changes in their data center budgets move NVIDIA’s sales.
*  **Supply limits**: Manufacturing capacity is limited and can’t be expanded quickly.
*   **Competition**: AMD, plus companies like Google and Amazon designing their own AI chips, could take share.
*   **Regulation and infrastructure**: U.S. export restrictions limit sales to some countries, and data centers need large amounts of power, which can slow how fast new ones get built.

## Data Sources & References

### Links to datasets used (original sources, not just download locations), & brief explanation of contents,

* https://epoch.ai/data-insights/hyperscaler-capex-trend -- aggregate data of capital expenditure since 2022 by the main players in the AI race. Epoch, in preparing the dataset, already accounted for the differences in how each company reports CAPEX spending and normalized it between the companies.

* https://www.sec.gov/edgar/browse/?CIK=0001045810 - SEC quarterly financial filings by NVIDIA as required because they're a public company. Data aggregated by Our World in Data here: https://ourworldindata.org/grapher/nvidia-quarterly-revenue-segment


* https://www.census.gov/construction/c30/historical_data.html -- historical and current data of public and private construction spend in various industries like office, commercial, etc. When datacenters are built, GPU orders aren't far behind.


### Citation of at least two credible references for the business context


*  NVIDIA Blog Announcing their partnership with TSMC in Arizona: https://blogs.nvidia.com/blog/nvidia-manufacture-american-made-ai-supercomputers-us/
*  NVIDIA SEC Report detailing their company, risks, and goals: https://www.sec.gov/Archives/edgar/data/1045810/000104581025000023/nvda-20250126.htm

## Data Acquisition & Loading - Nathan

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base = "https://raw.githubusercontent.com/mikahamelin-gif/IS-4487-final-project-nvidia/main/"

nvidia_url = base + "NVDA_Quarterly_Demand_Drivers.xlsx"
epoch_url = base + "epoch_hyperscaler_capex_2022_2025.csv"

census_url = "https://ourworldindata.org/grapher/monthly-spending-data-center-us.csv?v=1&csvType=full&useColumnShortNames=false"

In [12]:
nvidia = pd.read_excel(
    nvidia_url,
    sheet_name="AI Chip Demand Drivers"
)

nvidia.columns = nvidia.columns.str.strip()

nvidia["quarter"] = (
    nvidia["Year"].astype(int).astype(str)
    + "Q"
    + nvidia["Quarter Number"].astype(int).astype(str)
)

nvidia = nvidia.sort_values("quarter")

print(nvidia[[
    "quarter",
    "NVDA Revenue ($mm)",
    "NVDA Data Center Revenue ($mm)"
]].head())

print("NVIDIA Shape:", nvidia.shape)

  quarter  NVDA Revenue ($mm)  NVDA Data Center Revenue ($mm)
0  2016Q1                1305                             143
1  2016Q2                1428                             151
2  2016Q3                2004                             240
3  2016Q4                2173                             296
4  2017Q1                1937                             409
NVIDIA Shape: (42, 44)


In [13]:
epoch = pd.read_csv(epoch_url)

epoch.columns = epoch.columns.str.strip()

epoch["quarter"] = (
    epoch["Calendar Quarter"]
    .str.replace("CY", "", regex=False)
    .str.replace(" ", "", regex=False)
)

epoch = epoch.rename(columns={
    "MSFT Broad CapEx ($M)": "Microsoft_CapEx",
    "AMZN Broad CapEx ($M)": "Amazon_CapEx",
    "GOOG Broad CapEx ($M)": "Google_CapEx",
    "META Broad CapEx ($M)": "Meta_CapEx",
    "ORCL Broad CapEx ($M)": "Oracle_CapEx",
    "Total ($M)": "Total_Hyperscaler_CapEx"
})

capex_columns = [
    "Microsoft_CapEx",
    "Amazon_CapEx",
    "Google_CapEx",
    "Meta_CapEx",
    "Oracle_CapEx",
    "Total_Hyperscaler_CapEx"
]

for col in capex_columns:
    epoch[col] = pd.to_numeric(
        epoch[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

epoch = epoch.sort_values("quarter")

epoch["CapEx_YoY_Growth"] = (
    epoch["Total_Hyperscaler_CapEx"].pct_change(
        periods=4,
        fill_method=None
    ) * 100
)

epoch = epoch[
    ["quarter"] + capex_columns + ["CapEx_YoY_Growth"]
]

print(epoch.head())
print("Epoch Shape:", epoch.shape)

  quarter  Microsoft_CapEx  Amazon_CapEx  Google_CapEx  Meta_CapEx  \
0  2022Q1             6060         15117          9786        5556   
1  2022Q2             8011         15785          6828        7621   
2  2022Q3             6894         16509          7276        9375   
3  2022Q4             6872         16909          7595        9401   
4  2023Q1             7651         14215          6289        7062   

   Oracle_CapEx  Total_Hyperscaler_CapEx  CapEx_YoY_Growth  
0          1101                    37620               NaN  
1          1423                    39668               NaN  
2          1719                    41773               NaN  
3          2435                    43212               NaN  
4          2628                    37845          0.598086  
Epoch Shape: (16, 8)


In [16]:

census_url = base + "privtime.xlsx"

census = pd.read_excel(
    census_url,
    sheet_name="Private NSA",
    header=3,
    usecols="A,J"
)

census.columns = ["Date", "Construction_Spending"]

census["Date"] = (
    census["Date"].astype(str)
    .str.extract(r"([A-Za-z]{3}-\d{2})")[0]
)

census["Date"] = pd.to_datetime(
    census["Date"],
    format="%b-%y",
    errors="coerce"
)

census["Construction_Spending"] = pd.to_numeric(
    census["Construction_Spending"],
    errors="coerce"
)

census = census.dropna(subset=["Date"])

census["quarter"] = (
    census["Date"].dt.to_period("Q").astype(str)
)

construction = census.groupby("quarter").agg(
    Construction_Spending=("Construction_Spending", "sum"),
    Months_Reported=("Construction_Spending", "count")
).reset_index()

construction.loc[
    construction["Months_Reported"] != 3,
    "Construction_Spending"
] = np.nan

print(construction.tail(10))
print("Construction Shape:", construction.shape)


    quarter  Construction_Spending  Months_Reported
125  2024Q2                 8106.0                3
126  2024Q3                 9645.0                3
127  2024Q4                10325.0                3
128  2025Q1                10930.0                3
129  2025Q2                12545.0                3
130  2025Q3                12875.0                3
131  2025Q4                13387.0                3
132  2026Q1                13724.0                3
133  2026Q2                17069.0                3
134  2026Q3                23414.0                3
Construction Shape: (135, 3)


In [19]:

combined = pd.merge(
    nvidia,
    epoch,
    on="quarter",
    how="left"
)

combined = pd.merge(
    combined,
    construction,
    on="quarter",
    how="left"
)

combined = combined.sort_values("quarter").reset_index(drop=True)

print("Combined Dataset Shape:", combined.shape)

display(combined[[
    "quarter",
    "NVDA Revenue ($mm)",
    "NVDA Data Center Revenue ($mm)",
    "Total_Hyperscaler_CapEx",
    "Construction_Spending"
]].tail(15))


Combined Dataset Shape: (42, 53)


,quarter,NVDA Revenue ($mm),NVDA Data Center Revenue ($mm),Total_Hyperscaler_CapEx,Construction_Spending
27,2022Q4,6051,3600,43212.0,3349.0
28,2023Q1,7192,4300,37845.0,4078.0
29,2023Q2,13507,10300,36761.0,4647.0
30,2023Q3,18120,14500,40195.0,5377.0
31,2023Q4,22103,18400,46485.0,5893.0
32,2024Q1,26044,22600,49564.0,6721.0
33,2024Q2,30040,26300,60803.0,8106.0
34,2024Q3,35082,30800,66203.0,9645.0
35,2024Q4,39331,35600,83783.0,10325.0
36,2025Q1,44062,39100,82301.0,10930.0


In [20]:
combined.to_csv("NVDA_Combined_Demand_Drivers.csv", index=False)

print("Combined dataset saved successfully!")
print("Rows:", len(combined))
print("Columns:", len(combined.columns))

Combined dataset saved successfully!
Rows: 42
Columns: 53


## Data Exploration (EDA) -- Mikayla

In [ ]:
# initial inspection

In [ ]:
## summary statistics

In [ ]:
# visualizations of key trends & distributions

In [ ]:
# identification of potential issues (missing data, outliers, etc.)

## Data Cleaning & Preprocessing  -- Beck

In [ ]:
# Handling missing values, duplicates, and outliers


In [ ]:
# Encoding categorical variables


In [ ]:
# Scaling or normalization if required


In [ ]:
# Feature engineering (new calculated columns, transformations, aggregations)

## Modeling Approach - Mikayla

###Rationale for model selection (why you chose each algorithm)

### Description of 2–3 models to be tested

### Train/test split or cross-validation approach

## Model Implementation 1 - (model type) -- Nathan

In [ ]:
# Model Code

In [ ]:
# Model Tuning

In [ ]:
# Model Final Settings (post tuning)

In [ ]:
# Feature Selection Steps

 ## Model Implementation 2 - (model type) -- Beck

In [ ]:
# Model Code

In [ ]:
# Model Tuning

In [ ]:
# Model Final Settings (post tuning)

In [ ]:
# Feature Selection Steps

## Model Implementation 3 - (model type) -- Billy

In [ ]:
# Model Code

In [ ]:
# Model Tuning

In [ ]:
# Model Final Settings (post tuning)

In [ ]:
# Feature Selection Steps

## Model Evaluation -- Mikayla

In [ ]:
# Evaluate Model 1

In [ ]:
# Evaluate Model 2

In [ ]:
# Evaluate Model 3

In [ ]:
# Comparison of models

### Interpretation of model results & evaluation:


## Conclusions & Recommendations - Billy

### Best model choice and why

### Clear business recommendations

### Suggestions for next steps or further analysis

## Final Descriptive Charts - Nathan

Make sure all charts are labeled, correctly outputted, and have ocnsistent formatting

In [ ]:
# chart 1

In [ ]:
# chart 2

In [ ]:
# chart 3

In [ ]:
# chart 4

In [ ]:
# chart 5

In [ ]:
# chart 6 (optional)